# Day 1 — Website briefings with Gemini

Ali's course-based implementation of Ed Donner's first LLM engineering lab.

**Goal:** give a URL, retrieve its text, and display a factual Markdown briefing.

The order follows the instructor. Gemini replaces the OpenAI model, and our
variation uses a factual briefing instead of a humorous summary. The reusable
logic lives in `scraper.py` and `summarizer.py` at the repository root.

Select this project's `.venv` kernel. Run cells from top to bottom after a
kernel restart. Cells share a running Python session; their saved outputs do
not recreate variables. A full run makes two Gemini requests.


## 1. Imports

Locate the repository root so this notebook works when Cursor starts Python either in the root or in `notebooks/`. Import the same kinds of tools used in the course.


In [ ]:
import os
import sys
import inspect
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "pyproject.toml").is_file():
    project_root = project_root.parent
if not (project_root / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from the ai-engineering-lab repository.")
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
from summarizer import (
    DEFAULT_MODEL, GEMINI_BASE_URL, messages_for, summarize,
    system_prompt, user_prompt_prefix,
)

print(sys.executable)
print("Notebook ready")


## 2. Load and check the API key

Keep `.env` beside `pyproject.toml`, outside `.venv` and `notebooks`.
It must contain `GEMINI_API_KEY=...`; `GEMINI_MODEL` is optional.

`load_dotenv` loads settings into the Python process. `os.getenv` reads one
setting into a variable. These local checks do not validate the key with Google.
The instructor's `sk-proj-` prefix check applies to OpenAI keys, so it is omitted.


In [ ]:
load_dotenv(project_root / ".env", override=True)
api_key = os.getenv("GEMINI_API_KEY")

if not api_key or api_key == "your_actual_key_here":
    raise ValueError("Set GEMINI_API_KEY in the .env file at the project root.")
elif api_key.strip() != api_key:
    raise ValueError("Remove whitespace at either end of the key in .env.")
else:
    print("Gemini API key found and looks good so far!")

model = os.getenv("GEMINI_MODEL") or DEFAULT_MODEL
print("Model:", model)


## 3. First model call — prepare the message

`message` is a string. `messages` is a list containing one dictionary. Its role identifies the speaker, and its content holds the text. The last expression displays the list; this cell makes no API request.


In [ ]:
message = "Hello, Gemini! This is my first ever message to you! Hi!"
messages = [{"role": "user", "content": message}]
messages


## 4. First model call — send the message

`OpenAI` is the client class; `openai` is the variable holding our configured
helper. Google's address determines which provider receives the request.

This is the first actual model request. We use the Gemini model's default
reasoning settings. The final expression extracts the returned answer.


In [ ]:
from openai import APIStatusError

openai = OpenAI(
    api_key=api_key,
    base_url=GEMINI_BASE_URL,
    timeout=60.0,
    max_retries=0,
)
print("Model sent:", repr(model))

try:
    response = openai.chat.completions.create(model=model, messages=messages)
except APIStatusError as exc:
    print(f"HTTP {exc.status_code}")
    print(exc.response.text)
else:
    print(response.choices[0].message.content)

## 5. Fetch and inspect a website

The scraper downloads HTML and extracts readable text. The model receives
that text later. Change `url` to try another public webpage. This simple
scraper cannot render JavaScript or bypass a login page.


In [ ]:
url = "https://edwarddonner.com"
website = fetch_website_contents(url)
print(f"Extracted {len(website):,} characters")
print(website[:1500])


## 6. System prompt

This is the instruction governing the briefing. It is imported from `summarizer.py` so the notebook and terminal program use one shared version. Edit that file to change the style; restart the kernel and rerun the notebook to reload it.


In [ ]:
print(system_prompt)


## 7. User prompt

This introduces the page text. Missing information should be reported as missing. Only the first 20,000 extracted characters are sent to keep this first exercise small.


In [ ]:
print(user_prompt_prefix)


## 8. Assemble the messages

The course function `messages_for` combines the system instructions with a user message containing the website text. Inspect the function and the prepared messages before making a request.


In [ ]:
print(inspect.getsource(messages_for))
website_messages = messages_for(website)
print("Roles:", [message["role"] for message in website_messages])
print(website_messages[1]["content"][:1500])


## 9. Bring it together with summarize(url)

The shared `summarize` function fetches a page, prepares messages, calls Gemini, and returns text. Reading its source here keeps the instructor’s mechanism visible without maintaining a second implementation.


In [ ]:
print(inspect.getsource(summarize))


## 10. Display the summary

This is the second and final model request in a full notebook run. `display_summary` adds notebook presentation around the reusable summarization function.


In [ ]:
def display_summary(url):
    summary = summarize(url, model=model)
    display(Markdown(summary))
    return summary

summary = display_summary(url)


## 11. Save the generated briefing

Store the actual response locally as Markdown. The `outputs/` folder is ignored by Git. The notebook’s visible output can still be saved with Ctrl+S.


In [ ]:
output_path = project_root / "outputs" / "website_summary.md"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(f"# Website briefing\n\nSource: {url}\n\n{summary}\n", encoding="utf-8")
print("Saved:", output_path)


## 12. Run the same solution from a terminal

From the repository root:

```bat
uv run python summarizer.py https://edwarddonner.com --output outputs/website_summary.md
```

When revisiting this lab, try a different page, compare the briefing with the
source, and explain which changes belong to extraction, prompting, or display.
See `README.md` for setup, checks, limitations, and course attribution.
